In [ ]:
from pathlib import Path
parent_directory = Path.cwd().parent.parent
print(parent_directory) 
type(parent_directory)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
#reemplazar con la ruta de los archivos correcta
parent_directory = Path.cwd().parent.parent
FCST=pd.read_excel(parent_directory / "FCST Merck Abril 26.xlsx",sheet_name="Abril 2026")
PROD_DC_2=pd.read_excel(parent_directory / "PRODUCTOS DC.xlsx",sheet_name="Catalogo")

In [ ]:
"""""
Module: P&G Forecast extraction 2
Purpose: extraer ## de jeringas para todos los meses por producto
Date: 30/07/2026
Author: J.Gonzalez
"""
logbalanceo=[]
#Columnas importantes
id_cols = ['SKUMERCK']

#Calculo de mes y año actual
Month_today=pd.to_datetime("today").month
Year_today=pd.to_datetime("today").year
demand_today=pd.to_datetime(str(Year_today) + "-01-" + str(Month_today), format="%Y-%m-%d")
demand_today=str(demand_today)[0:10]

#print(FCST.columns)
#FCST.columns = [pd.to_datetime(col, errors='coerce').strftime('%Y-%m') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in FCST.columns]

matching_cols = []

for i in range(len(FCST.columns) - 1, -1, -1):  # from [-1] backwards
    col = FCST.columns[i]
    parsed = pd.to_datetime(col, errors='coerce')
    if pd.notna(parsed):
        matching_cols.append(col)
        if parsed.month <= Month_today and parsed.year <= Year_today:
            break  #
matching_cols = list(reversed(matching_cols))
#matching_cols = [pd.to_datetime(col, errors='coerce').strftime('%Y-%m') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in matching_cols]
#Union de columnas id con columna demanda mes actual
#keep_cols = id_cols + matching_cols
FCST_month = FCST[id_cols + matching_cols].copy()

FCST_month = FCST_month.dropna(subset=['SKUMERCK']).reset_index(drop=True)

#Forecast de jeringas por mes actual en csv
FCST_month.to_csv("FCST_month.csv", index=False)

""""
#extraccion columnas de demanda del mes actual

#en caso de encontrar dos columnas con la misma fecha conserva la segunda que ya incluye el calculo total de piezas.
if len(matching_cols) >= 2:
    matching_cols = [matching_cols[-1]]

Union de columnas id con columna demanda mes actual
if matching_cols:
    keep_cols = id_cols + matching_cols
    FCST_month = FCST[keep_cols].copy()
else:
    FCST_month = FCST[id_cols].copy()
FCST_month = FCST_month.rename(columns={FCST_month.columns[-1]: 'Demanda'})
"""

In [ ]:
"""""
Module: Produccion por linea 2
Purpose: Separa los productos por familia, y despues por linea que utiliza dentro de cada mes esa familia especifica
Date: 30/07/2026
Author: J.Gonzalez
"""
PROD_DC_2 = PROD_DC_2.merge(FCST_month, on='SKUMERCK', how='left')
PROD_DC_2.drop(['SKUMERCK'], axis=1, inplace=True)
#PROD_DC_2_months = [col for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']]

#dicccionario para la suma del group
agg_dict = {col: 'sum' for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']}

#Group by
PROD_DC_2 = PROD_DC_2.groupby(['Nombre granel', 'Linea Granel 1', 'Linea Granel 2'], as_index=False).agg(agg_dict)

#Una columna para identificar DC
PROD_DC_2['Linea Granel 1'] = np.where(
    (PROD_DC_2['Linea Granel 1'] == 1) & (PROD_DC_2['Linea Granel 2'] == 1),
    'DC2',
    'DC1')
PROD_DC_2.drop(['Linea Granel 2'], axis=1, inplace=True)
PROD_DC_2 = PROD_DC_2.rename(columns={'Linea Granel 1': 'DC'})
PROD_DC_2.to_csv("PROD_DC_2.csv", index=False)
print(PROD_DC_2)


In [ ]:
"""""
Module: Balanceo mensual
Purpose: balancea DC2 mandando su residuo a DC1 para cada familia en cada mes 
Date: 30/07/2026
Author: J.Gonzalez
"""

jeringas=115200

familias= PROD_DC_2['Nombre granel'].unique()
familias_con_dc2= PROD_DC_2['Nombre granel'].duplicated
familias_con_dc2= familias_con_dc2.unique()

print(familias_con_dc2)


for familias in familias_con_dc2:
    for col in agg_dict.keys():
        if PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0] % jeringas != 0:
            resdc2=PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0]%jeringas
            PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC1'].values[0]=PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC1'].values[0]+(resdc2*jeringas)
            PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0]=PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0]-(resdc2*jeringas)

            

"""
for col in PROD_DC_2.columns:
repeat until PROD_DC_2[col]%jeringas==0:
if PROD_DC_2[col]%jeringas>0:
    PROD_DC_2[col]=PROD_DC_2[col]-jeringas
    PROD_DC_1[col]=PROD_DC_1[col]+jeringas
"""

#revision_residuo_dc1=PROD_DC_2[PROD_DC_2['DC'] == 'DC1'].copy()
#revision_residuo_dc1=revision_residuo_dc1.agg(agg_dict)/jeringas
#revision_residuo_dc2=PROD_DC_2[PROD_DC_2['DC'] == 'DC2'].copy()
#revision_residuo_dc2=revision_residuo_dc2.agg(agg_dict)/jeringas
#revision_residuo_total= PROD_DC_2.groupby(['Nombre granel'], as_index=False).agg(agg_dict)
#revision_residuo_total=revision_residuo_total.agg(agg_dict)/jeringas

#revision_residuo_dc2.to_csv("revision_residuo_dc2.csv", index=False)
#revision_residuo_dc1.to_csv("revision_residuo_dc1.csv", index=False)
#revision_residuo_total.to_csv("revision_residuo_total.csv", index=False)

#modulo balance
